# dq_monitoring_logs — Squad 1
Gera e mantém a tabela de log de qualidade compartilhada por toda a squad,
a partir das colunas `regra_*` já calculadas na Silver de cada tabela.

**O que esta tabela guarda:** um registro por (tabela, regra, arquivo de
origem), com status PASS/FAIL, severidade, e contagem de falhas/total.

**Gravação incremental e segura:** usa `mode="append"`, só grava combinações
novas — nunca sobrescreve o que outras duplas da squad gravarem aqui.

In [0]:
# Célula 1 — setup
import os, uuid
from datetime import datetime, timezone
from dotenv import load_dotenv
from pyspark.sql import functions as F
import pandas as pd
from deltalake import DeltaTable, write_deltalake

load_dotenv(".env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = "internshipdatalake"
SQUAD_CONTAINER = "squad1"

lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"
dq_logs_path = f"{lakehouse_base}/dq_monitoring_logs"

def ler_delta_direto(caminho):
    dt = DeltaTable(caminho, storage_options=storage_options)
    return spark.createDataFrame(dt.to_pandas())

print("Setup concluído.")

`gerar_log_incremental`: calcula PASS/FAIL por regra e por micro-lote,
pulando combinações (tabela, regra, arquivo) que já existem na tabela —
evita duplicar quando o notebook roda de novo.

In [0]:
# Célula 2 — mapa de severidade e função de geração
SEVERIDADE = {
    "regra_01_sku_valido": "Critica", "regra_02_preco_positivo": "Critica",
    "regra_03_nome_preenchido": "Critica", "regra_04_categoria_existe": "Critica",
    "regra_05_unidade_valida": "Aviso", "regra_06_is_ativo_valido": "Critica",
    "regra_07_preco_dentro_limite": "Aviso", "regra_08_marca_preenchida": "Aviso",
    "regra_09_ativo_com_venda_90d": "Aviso", "regra_10_pertence_subcategoria": "Aviso",
    "regra_01_id_valido": "Critica", "regra_02_nome_preenchido": "Critica",
    "regra_03_pai_existe": "Critica", "regra_04_sem_ciclo_direto": "Critica",
    "regra_05_nome_unico_no_nivel": "Aviso", "regra_06_dois_niveis": "Critica",
    "regra_07_subcategoria_tem_produto": "Aviso", "regra_08_raiz_tem_subcategoria": "Aviso",
    "regra_09_nome_sem_caracteres_invalidos": "Aviso", "regra_10_total_raizes_confere": "Aviso",
}

def gerar_log_incremental(caminho_silver, nome_tabela, run_id, agora, ja_logados):
    df = ler_delta_direto(caminho_silver)
    colunas_regra = [c for c in df.columns if c.startswith("regra_")]
    linhas = []
    for regra in colunas_regra:
        agregado = (df.filter(F.col(regra).isNotNull())
            .groupBy("bronze_source_file")
            .agg(F.count("*").alias("total"), F.sum(F.when(~F.col(regra), 1).otherwise(0)).alias("falhas"))
            .collect())
        for row in agregado:
            if (nome_tabela, regra, row["bronze_source_file"]) in ja_logados:
                continue
            linhas.append((run_id, nome_tabela, regra,
                            "FAIL" if row["falhas"] > 0 else "PASS",
                            SEVERIDADE.get(regra, "Aviso"),
                            int(row["falhas"]), int(row["total"]), agora, row["bronze_source_file"]))
    return linhas

In [0]:
# Célula 3 — executa (idempotente: rodar de novo só grava o que for novo)
run_id = str(uuid.uuid4())
agora = datetime.now(timezone.utc)

try:
    dt_atual = DeltaTable(dq_logs_path, storage_options=storage_options)
    df_ja = dt_atual.to_pandas(columns=["tabela", "regra", "arquivo_origem"])
    ja_logados = set(zip(df_ja["tabela"], df_ja["regra"], df_ja["arquivo_origem"]))
except Exception:
    ja_logados = set()  # tabela ainda não existe — primeira carga

linhas_novas = (gerar_log_incremental(silver_produtos_path, "ecommerce_produtos", run_id, agora, ja_logados)
                + gerar_log_incremental(silver_categorias_path, "ecommerce_categorias", run_id, agora, ja_logados))

if linhas_novas:
    df_novo = pd.DataFrame(linhas_novas, columns=["run_id", "tabela", "regra", "status", "severidade",
                                                   "qtd_registros_falhos", "qtd_registros_total",
                                                   "timestamp_execucao", "arquivo_origem"])
    write_deltalake(dq_logs_path, df_novo, mode="append", storage_options=storage_options)
    print(f"{len(df_novo)} linhas novas gravadas")
else:
    print("nada novo para gravar")

Verificação: lê de volta do caminho (não da memória), confirma schema e
ausência de nulo, e quebra o resultado por regra — é essa quebra que mostra
qual regra específica está falhando, não só o agregado.

In [0]:
# Célula 4 — verificação + quebra por regra
df_check = DeltaTable(dq_logs_path, storage_options=storage_options).to_pandas()

print("Linhas:", len(df_check))
print("Nulos:\n", df_check.isnull().sum())
print("Status:", df_check["status"].unique(), "| Severidade:", df_check["severidade"].unique())

for tabela in df_check["tabela"].unique():
    print(f"\n=== {tabela} ===")
    display(df_check[df_check["tabela"] == tabela]
            .groupby("regra")["status"].value_counts().unstack(fill_value=0))